#### 01_demo.ipynb

- More demos showing how we work with diffent operators
- At the end of this notebook there is 2D CS(Sparse) Fourier Reconstrustruction Example where I concatenate several linear operators provided by this package

In [4]:

using Pkg

Pkg.activate("..") # From demos/ we go one level up to the project root
#Pkg.instantiate()  # Optional, but recommended to ensure all dependencies are installed

using MiniOps
using Random
using LinearAlgebra
using Plots
using Random 

  Activating project at `~/Dropbox/MiniOps`


In [5]:
# Test that 1D convolution and its adjoint (on the flight) pass the dot product test

function test_conv()

    h = randn(15)
    A = conv1d_op(h)

    x1 = randn(120) 
    y1 = A*x1
    
    y2 = randn(size(y1))
    x2 = A'*y2

println("<y1, y2> - <x1, x2>: ", y1'*y2 -x2'*x1)
println(A.name, " size : ", A.m, A.n)

end

test_conv()

<y1, y2> - <x1, x2>: -4.440892098500626e-15
conv1d size : -1-1


In [6]:
nt,nx,ny, nw ,nz = 20, 30, 40, 10, 5

# 1D case

S = sampling_op([2, 5, 9], nx)
y = S*randn(nx)
xa = S'*y
println("1D Case -->   size(y): ", size(y), "     size(xa):", size(xa))

# traces from a 2-D gather

R = trace_sampling_op([2, 5, 9], (nt, nx))
y = R*randn(nt,nx)
xa = R'*y
println("2D Case -->   size(y): ", size(y), "  size(xa):", size(xa))

# traces from a 3-D cube

R = trace_sampling_op([3, 11, 20], (nt, nx, ny))
y = R*randn(nt,nx,ny)
xa = R'*y
println("3D Case -->   size(y): ", size(y), "  size(xa):", size(xa))


# traces from a 3-D cube

R = sampling_op([3, 11, 20], (nt, nx, ny, nw))
y = R*randn(nt,nx,ny,nw)
xa = R'*y
println("4D Case -->   size(y): ", size(y), "  size(xa):", size(xa))

# traces from a 3-D cube

R = sampling_op([3, 11, 20], (nt, nx, ny, nw, nz))
y = R*randn(nt,nx,ny,nw,nz)
xa = R'*y
println("5D Case -->   size(y): ", size(y), "  size(xa):", size(xa))



1D Case -->   size(y): (3,)     size(xa):(30,)
2D Case -->   size(y): (20, 3)  size(xa):(20, 30)
3D Case -->   size(y): (20, 3)  size(xa):(20, 30, 40)
4D Case -->   size(y): (20, 3)  size(xa):(20, 30, 40, 10)
5D Case -->   size(y): (20, 3)  size(xa):(20, 30, 40, 10, 5)


In [7]:
# Test composition (concatenation) of several linear operators 

function test_composition()

  println("Testing do-product for A = S1*R*S2")
  println("S1 is scaling")
  println("S2 applies diag weights")
  println("R is sampling")

    nt, nx = 20, 30
   
    idx  = [1,2,3,20, 5, 10, 15, 25, 30]
    R = sampling_op(idx, (nt,nx))

    S1 = scaling_op(3.0)
    S2 = diag_op(ones(nt,nx))

    A = S1*R*S2
    
    x1 = randn(nt, nx)
    y1 = A*x1

    y2 = randn(size(y1))
    x2 = A'*y2

println("<x1,x2> = ", dot(x1,x2))
println("<y1,y2> = ", dot(y2,y1))
println("name = ", A.name, " Size: ",  A.m, A.n)
end
    
test_composition()

Testing do-product for A = S1*R*S2
S1 is scaling
S2 applies diag weights
R is sampling
<x1,x2> = 38.27887370537929
<y1,y2> = 38.27887370537929
name = scaling*trace_sampling*diag Size: -1-1


In [36]:
# Multiple dispatch is used to generate two FFT operators (size planned and size unplanned)

A = fft_op(randn(ComplexF64,20,20))
B = fft_op()

x1 = randn(ComplexF64,20,20)
y1 = A*x1

y2 = randn(ComplexF64,20,20)
x2 = A'*y2

println("<x1,x2> = ", round(dot(x1,x2), digits=8))
println("<y1,y2> = ", round(dot(y1,y2), digits=8))

A = with_shape(A, x1)
println(  "m : ", A.name,  " n : ", A.n)
println(  "m : ", B.name,  " n : ", B.n)

B = with_shape(B, x1)
println(  "m : ", B.name,  " n : ", B.n)

<x1,x2> = 12.68874495 + 22.13047498im
<y1,y2> = 12.68874495 + 22.13047498im
m : fft_planned n : 400
m : fft_op n : -1
m : fft_op n : 400


In [37]:
# Dot product test for the multichannel convolution and it's adjoint 
function test_conv_multichannel()

    wavelet = rand(10)
    A = conv1d_cols_op(wavelet)

    x1 = randn(200,100)
    y1 = A*x1

    y2 = randn(size(y1))
    x2 = A'*y2

println("<x1,x2> = ", dot(x1,x2))
println("<y1,y2> = ", dot(y1,y2))
    
println("name : ", A.name," m : ", A.m, " n : ", A.n)
B = with_shape(A, x1)
println("name : ", B.name," m : ", B.m, " n : ", B.n)
end
test_conv_multichannel()

<x1,x2> = 203.86331456583554
<y1,y2> = 203.8633145658356
name : conv1d_cols m : -1 n : -1
name : conv1d_cols m : 20900 n : 20000
